# DynaLoRA experiments (Colab, GPU)

Runs the full pipeline of the paper: trains LoRA adapters for each corpus configuration, generates under DynaLoRA, weight-space scaling, classifier-free guidance and the temperature baselines, scores the generations (perplexity under GPT-2 large, domain classifier trained on held-out reference texts, sentiment classifier, lexical metrics), and computes bootstrap intervals and LLM-judge ratings.

Runtime: select a GPU (a T4 is enough). Expected time: about 2 to 3 hours for GPT-2 small on five configurations plus Qwen2.5-0.5B on one, and about 2 hours for the two LLM judges.

In [ ]:
!nvidia-smi -L
!git clone -b paper https://github.com/alexroehrs/dynalora.git /content/dynalora
%cd /content/dynalora
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao  # Colab ships a torchao version that the current peft rejects

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT='/content/drive/MyDrive/dynalora_results'
os.makedirs(OUT, exist_ok=True)
print(OUT)

## 1. GPT-2 small, five corpus configurations
`imdb_neg` is the under-trained contrast (negative reviews only, learning rate 5e-6, unmasked padding). The others use balanced IMDB, balanced IMDB without HTML tags, Yelp and PubMed abstracts, with the standard LoRA learning rate and masked padding. Results are written incrementally, so the cell can be re-run after a disconnection and skips what is done.

In [ ]:
!python scripts/run_all.py --out $OUT/gpt2 --models gpt2 --configs imdb_neg imdb_bal imdb_bal_nohtml yelp pubmed --n_samples 20

## 2. Qwen2.5-0.5B on balanced IMDB

In [ ]:
!python scripts/run_all.py --out $OUT/qwen --models Qwen/Qwen2.5-0.5B --configs imdb_bal --n_samples 20

## 3. Bootstrap intervals and paired tests

In [ ]:
!python scripts/bootstrap_ci.py --inputs "$OUT/*/metrics__*.csv.gz" --out $OUT/stats

## 4. LLM judges (4-bit, stratified sample of 50 texts per cell)

In [ ]:
!pip install -q bitsandbytes
!python scripts/llm_judge.py --inputs $OUT/gpt2/generations_all_metrics.csv $OUT/qwen/generations_all_metrics.csv --out $OUT/judge

## 5. Package the results

In [ ]:
!cd $OUT && zip -qr /content/dynalora_results.zip . -x '*/adapters/*'
from google.colab import files
files.download('/content/dynalora_results.zip')